In [2]:
# traplining identifiaction
# import sessdf
from utils.opconNosepokeFunctions import *
from utils.supplementaryFunctions import *
from scipy.optimize import minimize
from scipy.stats import entropy
from scipy.stats import ttest_rel
%load_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np

from utils.plotSettings import *
%matplotlib qt

from utils.dfLoading import read_df, remove_sessions_with_few_duplicate_trials
sessdf = read_df('S:/fileTransferFromWindows/sessdf-2026-04-23.csv', task_type = ['unstr', 'str', 'dms', 'dls', 'sham'])

# Apply to sessdf
sessdf = remove_sessions_with_few_duplicate_trials(sessdf, max_duplicates = 20)
from utils.dfLoading import add_block_groups
sessdf = add_block_groups(sessdf)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


c:\Users\dlab\banditAnalysis\utils\dfLoading.py:44: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df.assigned.replace({True: 'str', False: 'unstr'}, inplace = True)
c:\Users\dlab\banditAnalysis\utils\dfLoading.py:48: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when d

removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed
removed


In [3]:
block_an_dict = {'unblocked': ['test05022023', 'Blissey', 'Chikorita', 'Darkrai', 'Eevee',
       'Goldeen', 'Hoppip', 'Inkay', 'Jirachi', 'Kirlia', 'Mesprit',
       'Nidorina', 'Oddish', 'Phione', 'Quilava', 'Raltz', 'Shinx',
       'Togepi', 'Umbreon', 'Vulpix', 'Xatu', 'Yanma', 'Zacian',
       'Alakazam', 'Bayleef', 'Cresselia', 'Haxorus'],
       'cued':['Emolga', 'Giratina',
              #   'Haxorus',
                 'Ivysaur',
              'Jigglypuff', 'Lugia', #'Ninetales', 'Onix',
              'Pichu'],
       'uncued' : ['Quaxly', 'Torchic', 'Uxie', 'Vanillish',
              'Whismur', 'Xerneas', 'Yamper', 'Zorua']}
an_dict = {
    'unstr'  : ['test05022023', 'Blissey', 'Chikorita', 'Darkrai', 'Eevee', 
                'Goldeen', 'Hoppip', 'Inkay', 'Jirachi', 'Kirlia', 'Mesprit',
                'Nidorina', 'Oddish', 'Phione', 'Quilava', 'Raltz', 'Shinx',
                'Togepi', 'Umbreon', 'Vulpix', 'Xatu', 'Yanma', 'Zacian',
                'Alakazam', 'Bayleef', 'Cresselia', 'Emolga', 'Giratina',
                'Haxorus', 'Ivysaur', 'Jigglypuff', 'Lugia', #'Ninetales', 'Onix',
                'Pichu', 'Quaxly', 'Sableye', 'Torchic', 'Uxie', 'Vanillish',
                'Whismur', 'Xerneas', 'Yamper', 'Zorua'],
    'dls'    : ['Blissey', 'Darkrai', 'Inkay' , 'Mesprit', 'Raltz', 'Zacian', 'Alakazam', 'Quilava', 'Torchic', 'Uxie', 'Vanillish', 'Yamper', 'Zorua'],
    'dms'    : ['Hoppip', 'Kirlia', 'Nidorina', 'Togepi', 'Xatu', 'Yanma', 'Giratina', 'Ivysaur', 'Lugia', 'Whismur', 'Quaxly'],
    'sham'   : ['test05022023', 'Jirachi', 'Goldeen', 'Phione', 'Umbreon', 'Vulpix', 'Emolga', 'Haxorus', 'Jigglypuff'],
    'ds_2stage' : ['Chikorita', 'Eevee'],
    'ds_4stage': ['Hoppip', 'Blissey'],
    'ds'     : ['Chikorita', 'Eevee', 'Hoppip', 'Blissey'],
    'str'    : ['test05022023', 'Blissey', 'Chikorita', 'Darkrai', 'Eevee'],
    'hsb_str' : ['HSB_F_006', 'HSB_F_007', 'HSB_F_008', 'HSB_F_009', 'HSB_F_010', 'HSB_F_011'],
    'oe_drive' : ['Shinx', 'Dratini', 'Bayleef', 'Onix', 'Pichu', 'Xerneas', 'Bidoof', 'Dwebble']
}

In [6]:
# for every animal
parula = get_parula_cmap()
num_trials_start = 0
num_trials_end = 20
# calculate the port prob in the first 10 ports across sessions >100 trials long
dataset = (sessdf[(sessdf.task == 'unstr') & (sessdf.block_group == 1) & (sessdf.animal.isin(block_an_dict['cued']+block_an_dict['uncued']))& (sessdf.sess_bin>=4)]
                               .groupby(['animal', 'session'])
                               .filter(lambda x: x.reward.size >= 100)).groupby(['animal', 'session']).head(num_trials_end)

# at every trial 
mat = np.zeros((dataset.animal.unique().shape[0], num_trials_end - num_trials_start, 4))

plt.figure(figsize = (20, 10))
for ind, an in enumerate(dataset.animal.unique()):
    for t, trial in enumerate(range(num_trials_start, num_trials_end)):
        hist = dataset[dataset.animal == an].groupby('session').nth(trial).port.value_counts(normalize = True).sort_index()
        hist = hist.reindex(index = [1,2,3,4], fill_value = 0)            
        mat[ind, t, :] = hist
    ax = plt.subplot(5, 8, ind+1)
    ax.imshow(mat[ind, :, :].T, aspect='auto', vmax = 0.5, vmin = 0, cmap = parula,)
    ax.set_yticks([0,1,2,3], [1,2,3,4])
    # ax.set_yticks()
    ax.set_title(an)
    # ax.set_xlabel(f'trials {num_trials_start}-{num_trials_end}')
plt.tight_layout()

In [16]:
# pre- and post- lesion comparison

# for every animal
parula = get_parula_cmap()
num_trials_start = 0
num_trials_end = 20
# calculate the port prob in the first 10 ports across sessions >100 trials long
dataset_pre = (sessdf[(sessdf.task == 'unstr') & (sessdf.block_group == 1) & (sessdf.animal.isin(an_dict['dls'])) & (sessdf.week_task>=6)]
                               .groupby(['animal', 'session'])
                               .filter(lambda x: x.reward.size >= 100)).groupby(['animal', 'session']).head(num_trials_end)
dataset_post = (sessdf[(sessdf.task == 'dls') & (sessdf.block_group == 1) & (sessdf.animal.isin(an_dict['dls'])) & (sessdf.week_task<=2)]
                               .groupby(['animal', 'session'])
                               .filter(lambda x: x.reward.size >= 100)).groupby(['animal', 'session']).head(num_trials_end)


# at every trial 
mat_pre = np.zeros((dataset_pre.animal.unique().shape[0], num_trials_end - num_trials_start, 4))
mat_post = np.zeros((dataset_pre.animal.unique().shape[0], num_trials_end - num_trials_start, 4))
pn = 1
plt.figure(figsize = (20, 10))
for ind, an in enumerate(dataset_pre.animal.unique()):
    for t, trial in enumerate(range(num_trials_start, num_trials_end)):
        hist_pre = dataset_pre[dataset_pre.animal == an].groupby('session').nth(trial).port.value_counts(normalize = True).sort_index()
        hist_pre = hist_pre.reindex(index = [1,2,3,4], fill_value = 0)            
        mat_pre[ind, t, :] = hist_pre

        hist_post = dataset_post[dataset_post.animal == an].groupby('session').nth(trial).port.value_counts(normalize = True).sort_index()
        hist_post = hist_post.reindex(index = [1,2,3,4], fill_value = 0)            
        mat_post[ind, t, :] = hist_post
    ax = plt.subplot(5, 8, pn)
    ax.imshow(mat_pre[ind, :, :].T, aspect='auto', vmax = 0.5, vmin = 0, cmap = 'viridis',)
    ax.set_yticks([0,1,2,3], [1,2,3,4])
    # ax.set_yticks()
    ax.set_title(an)
    pn+=1
    # ax.set_xlabel(f'trials {num_trials_start}-{num_trials_end}')
    ax = plt.subplot(5,8,pn)
    ax.imshow(mat_post[ind, :, :].T, aspect='auto', vmax = 0.5, vmin = 0, cmap = 'viridis',)
    ax.set_yticks([0,1,2,3], [1,2,3,4])
    ax.set_title(an)
    pn+=1
plt.tight_layout()